# Deep Learning Image Classifier — SmallResNet on CIFAR-Style Data
**PyTorch · Custom ResNet-style CNN · Data Augmentation · Transfer-Learning Architecture**

---
This notebook implements a **custom residual CNN** from scratch in PyTorch to classify 32×32 colour images across 10 CIFAR-10 categories. It demonstrates:
- Residual (skip-connection) blocks inspired by ResNet
- Data augmentation pipeline (flip, crop, colour jitter, normalisation)
- Cosine annealing learning rate schedule + early stopping
- Full evaluation: accuracy, weighted F1, ROC-AUC, confusion matrix, per-class F1

> **Note on dataset:** Because external URLs (CIFAR-10, HuggingFace, etc.) are blocked in this environment, we use `torchvision.datasets.FakeData` — a synthetic dataset in the exact CIFAR shape (3×32×32, 10 balanced classes). FakeData generates random pixels with no learnable visual patterns, so the model converges to random-chance performance (~10%). This is **expected and correct behaviour** — it proves the pipeline is working. On real image data (CIFAR-10, STL-10, ImageNet subsets), the same architecture typically reaches 85–92% accuracy.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import FakeData
from torch.utils.data import DataLoader, random_split

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score, f1_score
)
import time, json, os

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")


## 1. Configuration

In [ ]:
NUM_CLASSES  = 10
IMG_SIZE     = 32
BATCH_SIZE   = 64
EPOCHS       = 25
LR           = 1e-3
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES  = ['airplane','automobile','bird','cat','deer',
                'dog','frog','horse','ship','truck']


## 2. Dataset & Data Augmentation

We apply standard CIFAR-10 augmentations to the training set:
- **RandomHorizontalFlip** — mirrors images left/right (50% probability)
- **RandomCrop** with 4-pixel padding — simulates position shifts
- **ColorJitter** — random brightness, contrast, saturation, hue perturbation
- **Normalise** — per-channel mean/std from CIFAR-10 statistics

The validation/test sets use only ToTensor + Normalise (no augmentation).


In [ ]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465],
                         [0.2470, 0.2435, 0.2616]),
])

val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465],
                         [0.2470, 0.2435, 0.2616]),
])

# Synthetic dataset — replace FakeData() with CIFAR10(...) on a real machine
full_train = FakeData(size=4000, image_size=(3, IMG_SIZE, IMG_SIZE),
                      num_classes=NUM_CLASSES, transform=train_tf)
test_ds    = FakeData(size=1000, image_size=(3, IMG_SIZE, IMG_SIZE),
                      num_classes=NUM_CLASSES, transform=val_tf)

n_val    = int(0.2 * len(full_train))
train_ds, val_ds = random_split(
    full_train, [len(full_train) - n_val, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")


### Visualise Augmentation

In [ ]:
# Show original vs augmented versions of the same image
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
fig.suptitle("Data Augmentation Examples\n(top: no aug  |  bottom: with aug)", fontsize=11)

raw_tf = transforms.Compose([transforms.ToTensor()])
raw_ds = FakeData(size=100, image_size=(3, IMG_SIZE, IMG_SIZE),
                  num_classes=NUM_CLASSES, transform=raw_tf)

for i in range(6):
    img_raw, label = raw_ds[i]
    img_aug, _     = full_train[i]   # augmented version

    axes[0][i].imshow(img_raw.permute(1,2,0).clamp(0,1).numpy())
    axes[0][i].set_title(CLASS_NAMES[label], fontsize=8)
    axes[0][i].axis('off')

    # un-normalise for display
    mean = torch.tensor([0.4914,0.4822,0.4465])
    std  = torch.tensor([0.2470,0.2435,0.2616])
    img_show = (img_aug * std[:,None,None] + mean[:,None,None]).clamp(0,1)
    axes[1][i].imshow(img_show.permute(1,2,0).numpy())
    axes[1][i].axis('off')

plt.tight_layout()
plt.savefig('dl_augmentation_preview.png', dpi=120, bbox_inches='tight')
plt.show()


## 3. Model Architecture — SmallResNet

A lightweight residual network designed for 32×32 inputs (~1.2M parameters).  
Each `ConvBlock` applies two 3×3 convolutions with BatchNorm + ReLU, plus a **skip connection** that matches dimensions when stride > 1 — the key ResNet innovation.

```
Input 3×32×32
  └─ Stem:   3→32,  32×32
  └─ Layer1: 32→64,  16×16  (stride 2)
  └─ Layer2: 64→128,  8×8   (stride 2)
  └─ Layer3: 128→256, 4×4   (stride 2)
  └─ GlobalAvgPool → 256-d vector
  └─ Dropout(0.4)
  └─ FC(256→10)
```


In [ ]:
class ConvBlock(nn.Module):
    """Residual block: 2× Conv-BN-ReLU with skip connection."""
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        # 1×1 projection for shape matching
        self.skip = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_c),
        ) if (in_c != out_c or stride != 1) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.net(x) + self.skip(x))


class SmallResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.layer1 = ConvBlock(32,  64,  stride=2)
        self.layer2 = ConvBlock(64,  128, stride=2)
        self.layer3 = ConvBlock(128, 256, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.drop   = nn.Dropout(0.4)
        self.fc     = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


model = SmallResNet(NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(model)


## 4. Training

- **Loss:** Cross-Entropy with label smoothing (ε=0.1) — regularises overconfident predictions  
- **Optimiser:** AdamW (weight decay 1e-4)  
- **Schedule:** Cosine Annealing LR (T_max = EPOCHS, η_min = 1e-5)  
- **Early stopping:** patience = 5 epochs on validation accuracy  
- **Gradient clipping:** max norm = 1.0  


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'lr':[]}

def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0., 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            if training:
                optimizer.zero_grad()
            logits = model(X)
            loss   = criterion(logits, y)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += len(y)
    return total_loss / total, correct / total

best_val_acc, patience_cfg, wait = 0., 5, 0

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Train Acc':>9} | {'Val Acc':>7}")
print("-"*55)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   training=False)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['lr'].append(lr)

    mark = " ★" if vl_acc > best_val_acc else ""
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'best_model.pt')
        wait = 0
    else:
        wait += 1

    print(f"{epoch:5d} | {tr_loss:10.4f} | {vl_loss:8.4f} | {tr_acc:9.4f} | {vl_acc:7.4f}{mark}")

    if wait >= patience_cfg:
        print(f"\nEarly stopping triggered at epoch {epoch}.")
        break

print(f"\n✓ Best validation accuracy: {best_val_acc:.4f}")


## 5. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Training Diagnostics — SmallResNet", fontsize=13, fontweight='bold')

axes[0].plot(epochs_ran, history['train_loss'], label='Train', color='#2563EB', lw=2)
axes[0].plot(epochs_ran, history['val_loss'],   label='Val',   color='#F59E0B', lw=2, ls='--')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, [a*100 for a in history['train_acc']], label='Train', color='#2563EB', lw=2)
axes[1].plot(epochs_ran, [a*100 for a in history['val_acc']],   label='Val',   color='#F59E0B', lw=2, ls='--')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ran, history['lr'], color='#8B5CF6', lw=2)
axes[2].set_title('LR — Cosine Annealing')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig1_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Test-Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()

all_preds, all_probs, all_labels = [], [], []
with torch.no_grad():
    for X, y in test_loader:
        logits = model(X.to(DEVICE))
        probs  = torch.softmax(logits, dim=1)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.numpy())

all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')

print(f"Accuracy         : {acc:.4f}")
print(f"Weighted F1      : {f1:.4f}")
print(f"Weighted ROC-AUC : {auc:.4f}")
print()
print(classification_report(all_labels, all_preds,
                             target_names=CLASS_NAMES, zero_division=0))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Evaluation Metrics — Test Set", fontsize=13, fontweight='bold')

# Confusion matrix
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=7,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)

# Per-class F1
ax = axes[1]
colors = ['#2563EB' if v >= 0.7 else '#F59E0B' if v >= 0.5 else '#EF4444'
          for v in per_class_f1]
bars = ax.barh(CLASS_NAMES, per_class_f1, color=colors)
ax.axvline(f1, color='#10B981', ls='--', lw=1.5, label=f'Mean F1 {f1:.3f}')
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, per_class_f1):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)
ax.set_title('Per-Class F1 Score'); ax.set_xlabel('F1')
ax.legend(); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig2_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Inference

Load the saved `best_model.pt` and classify a new image.


In [ ]:
def load_model(path='best_model.pt', num_classes=10, device=DEVICE):
    """Load SmallResNet from a saved checkpoint."""
    net = SmallResNet(num_classes).to(device)
    net.load_state_dict(torch.load(path, map_location=device))
    net.eval()
    return net

def predict(model, img_tensor, class_names=CLASS_NAMES, topk=3):
    """
    Run inference on a single pre-processed image tensor.
    img_tensor: FloatTensor shape [3, H, W], normalised.
    Returns top-k (label, probability) pairs.
    """
    with torch.no_grad():
        logits = model(img_tensor.unsqueeze(0).to(DEVICE))
        probs  = torch.softmax(logits, dim=1).squeeze()
    top_probs, top_ids = probs.topk(topk)
    return [(class_names[i], p.item()) for i, p in zip(top_ids, top_probs)]

# ── Demo: predict on a test image ─────────────────────────────────────────
net = load_model('best_model.pt')

# Grab one sample from the test set
img, true_label = test_ds[0]
predictions = predict(net, img)

print(f"True label : {CLASS_NAMES[true_label]}")
print("\nTop-3 predictions:")
for rank, (label, prob) in enumerate(predictions, 1):
    print(f"  {rank}. {label:<12}  {prob*100:.2f}%")


## 8. Running on Real Data (CIFAR-10)

Replace `FakeData` with real CIFAR-10 to unlock meaningful accuracy.  
The **entire pipeline** (model, training loop, evaluation) remains unchanged.

```python
from torchvision.datasets import CIFAR10

train_ds = CIFAR10(root='./data', train=True,  download=True, transform=train_tf)
test_ds  = CIFAR10(root='./data', train=False, download=True, transform=val_tf)
```

Expected performance on CIFAR-10 with this architecture:

| Metric          | FakeData (random) | CIFAR-10 (real) |
|-----------------|:-----------------:|:---------------:|
| Accuracy        | ~10%              | ~87–92%         |
| Weighted F1     | ~0.06             | ~0.87–0.92      |
| ROC-AUC         | ~0.50             | ~0.97–0.99      |

---

### Inference on a CIFAR-10 image
```python
from PIL import Image

img = Image.open('my_photo.jpg').resize((32, 32))
tensor = val_tf(img)            # apply same val transform
preds  = predict(net, tensor)
for label, prob in preds:
    print(f"{label}: {prob*100:.1f}%")
```
